# Model Monitoring Drift Checks

## Objective

Build an empirical monitoring snapshot from the held-out processed test split. The notebook uses the selected model and the shared `yield_risk.monitoring.build_monitoring_summary` utility to compare deterministic reference and current batches for missingness drift, feature distribution drift, prediction score drift, and high-risk-rate drift.

In [1]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    """Find the nearest parent directory containing src/."""
    for candidate in (start, *start.parents):
        if (candidate / "src").is_dir():
            return candidate
    raise RuntimeError("Could not locate repository root containing src/.")


try:
    start_dir = Path(__file__).resolve().parent
except NameError:
    start_dir = Path.cwd().resolve()

repo_root = find_repo_root(start_dir)
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

repo_root

WindowsPath('C:/Users/Virgil/Code/semiconductor-yield-root-cause')

In [2]:
import joblib
import pandas as pd

from yield_risk.config import load_config
from yield_risk.monitoring import build_monitoring_summary

## Load Artifacts

Load the run configuration, processed held-out test data, selected model, and model comparison table from repository-root-relative paths.

In [3]:
cfg = load_config(repo_root / "configs" / "config.yaml")

test = pd.read_csv(repo_root / cfg.paths.splits_dir / "test.csv")
sensor_cols = [column for column in test.columns if column.startswith("sensor_")]
model = joblib.load(repo_root / cfg.paths.models_dir / "selected_model.joblib")
model_comparison = pd.read_csv(repo_root / cfg.paths.reports_dir / "model_comparison.csv")

selected = model_comparison.loc[model_comparison["selected"]].iloc[0]
threshold = float(selected["opt_threshold"])

artifact_summary = pd.DataFrame(
    [
        {
            "test_rows": len(test),
            "sensor_features": len(sensor_cols),
            "selected_model": selected["model"],
            "operating_threshold": threshold,
        }
    ]
)
artifact_summary

,test_rows,sensor_features,selected_model,operating_threshold
0,314,198,random_forest,0.08


In [4]:
model_comparison

,model,cv_pr_auc_mean,test_pr_auc,test_roc_auc,test_recall,test_precision,opt_threshold,expected_cost,selected
0,dummy,0.066003,0.065962,0.489680,0.047619,0.047619,0.01,220.0,False
1,logistic_regression,0.181964,0.209589,0.669267,0.571429,0.130435,0.47,170.0,False
2,random_forest,0.216511,0.193402,0.758411,0.571429,0.171429,0.08,148.0,True
3,xgboost,0.208186,0.261061,0.801560,0.666667,0.222222,0.03,119.0,False


## Deterministic Monitoring Batches

Split the held-out test data into fixed reference and current halves, then score both batches with the selected model.

In [5]:
midpoint = len(test) // 2
reference = test.iloc[:midpoint].copy()
current = test.iloc[midpoint:].copy()

reference_scores = model.predict_proba(reference[sensor_cols])[:, 1]
current_scores = model.predict_proba(current[sensor_cols])[:, 1]

pd.DataFrame(
    [
        {
            "reference_rows": len(reference),
            "current_rows": len(current),
            "reference_score_mean": reference_scores.mean(),
            "current_score_mean": current_scores.mean(),
        }
    ]
)

,reference_rows,current_rows,reference_score_mean,current_score_mean
0,157,157,0.056367,0.066493


## Monitoring Summary

Build all monitoring sections through the shared library utility.

In [6]:
summary = build_monitoring_summary(
    reference,
    current,
    sensor_cols,
    reference_scores,
    current_scores,
    high_risk_threshold=threshold,
)
summary

MonitoringSummary(missingness=MissingnessDriftSummary(results=[MissingnessDriftResult(feature='sensor_000', reference_missing_rate=0.0, current_missing_rate=0.0, missing_rate_delta=0.0, alert=False), MissingnessDriftResult(feature='sensor_001', reference_missing_rate=0.0, current_missing_rate=0.0, missing_rate_delta=0.0, alert=False), MissingnessDriftResult(feature='sensor_002', reference_missing_rate=0.0, current_missing_rate=0.0, missing_rate_delta=0.0, alert=False), MissingnessDriftResult(feature='sensor_003', reference_missing_rate=0.0, current_missing_rate=0.0, missing_rate_delta=0.0, alert=False), MissingnessDriftResult(feature='sensor_004', reference_missing_rate=0.0, current_missing_rate=0.0, missing_rate_delta=0.0, alert=False), MissingnessDriftResult(feature='sensor_006', reference_missing_rate=0.0, current_missing_rate=0.0, missing_rate_delta=0.0, alert=False), MissingnessDriftResult(feature='sensor_012', reference_missing_rate=0.0, current_missing_rate=0.0, missing_rate_del

### Missingness Drift Alerts

In [7]:
missingness_alerts = summary.missingness.to_frame().query("alert")
missingness_alerts.head(10)

,feature,reference_missing_rate,current_missing_rate,missing_rate_delta,alert


### Top Feature Drift Rows

In [8]:
summary.features.to_frame().sort_values(
    ["alert", "mean_delta", "ks_statistic"], ascending=[False, False, False]
).head(10)

,feature,reference_mean,current_mean,mean_delta,reference_std,current_std,std_delta,reference_median,current_median,median_delta,ks_statistic,ks_p_value,alert
83,sensor_161,3685.452229,4350.528662,665.076433,3507.472257,4551.772345,1044.300088,2449.0000,2620.0000,171.0000,0.076433,0.723123,True
84,sensor_162,4618.643312,4871.369427,252.726115,6621.617351,7142.434198,520.816847,1378.0000,1628.0000,250.0000,0.082803,0.628452,True
14,sensor_023,-3920.488322,-3747.736199,172.752123,1086.107959,1477.316253,391.208293,-3828.3333,-3883.0000,54.6667,0.089172,0.535002,True
15,sensor_024,-180.482484,-317.159236,136.676752,3049.002150,2641.444839,407.557312,51.7500,-80.5000,132.2500,0.095541,0.447217,True
12,sensor_021,-5684.404459,-5584.955414,99.449045,569.054726,594.147770,25.093043,-5558.2500,-5514.7500,43.5000,0.121019,0.187460,True
56,sensor_090,8763.273290,8838.578108,75.304818,399.428785,377.300807,22.127978,8790.2500,8825.4351,35.1851,0.114650,0.237933,True
82,sensor_160,505.687898,578.363057,72.675159,491.155856,652.638634,161.482778,414.0000,428.0000,14.0000,0.089172,0.535002,True
149,sensor_468,203.861509,250.629931,46.768422,209.807125,233.272793,23.465668,139.1762,168.0216,28.8454,0.140127,0.084701,True
116,sensor_225,1016.582761,1063.298097,46.715336,412.529885,480.348843,67.818959,967.2998,973.7998,6.5000,0.063694,0.889581,True
158,sensor_484,235.108815,189.156053,45.952762,230.757558,197.698251,33.059306,153.7012,122.9050,30.7962,0.108280,0.297913,True


### Prediction Drift Summary

In [9]:
pd.DataFrame([summary.predictions.__dict__])

,reference_mean,current_mean,mean_delta,reference_median,current_median,median_delta,reference_std,current_std,std_delta,reference_p90,current_p90,p90_delta,ks_statistic,ks_p_value,alert
0,0.056367,0.066493,0.010126,0.049725,0.05654,0.006815,0.03261,0.039966,0.007356,0.103052,0.118917,0.015865,0.133758,0.111833,True


### High-Risk-Rate Drift Summary

In [10]:
pd.DataFrame([summary.high_risk_rate.__dict__])

,threshold,reference_rate,current_rate,rate_delta,rate_delta_threshold,alert
0,0.08,0.191083,0.254777,0.063694,0.05,True


## Monitoring Interpretation

This notebook uses historical SECOM data and anonymous sensors. Missingness is stable across the reference and current batches, while feature drift is broad under the raw thresholds. Prediction-score and high-risk-rate checks are triggered, so the next step is calibrating monitoring thresholds against an operating policy before wiring the checks into a production feedback loop.